# CMZ — Behavior detection results (day & night)

Proof that the deployed system works. For each behavior (**Eating, Locomotion, Resting**), in
**daylight and infrared**, it tracks the civet and labels its behavior, producing an **observation
sheet** (time → Active/Inactive → behavior) and an **annotated preview**.

**Pipeline:** custom YOLO-seg civet detector + ByteTrack tracker → 3-class ResNet behavior
classifier → motion gate + food-tray rule.

**Put these on Drive first (`CMZprojectphase2/`):**
- `models/civet_bowl_seg_best.pt` — the YOLO civet detector (rename `best.pt` to this)
- `models/model.pt` — the 3-class behavior ResNet
- `data/raw/cam3/A03_20260405140327.mp4` — daytime video
- `data/raw/cam3/A03_20260405000000.mp4` — night video

Run the two cells below top to bottom.

In [ ]:
# 1) Setup: mount Drive, pull the code, install deps
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CMZprojectphase2")
assert PROJECT_ROOT.is_dir(), f"Add a shortcut to the shared {PROJECT_ROOT.name} under My Drive"
drive_root = PROJECT_ROOT

repo = Path("/content/cmz-project"); branch = "feat/sam3-pixel-annotation"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{branch}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", branch,
                    "https://github.com/Siwali/cmz-project.git", str(repo)], check=True)
PIPE = repo / "pipeline"; sys.path.insert(0, str(PIPE))

!pip -q install ultralytics imageio-ffmpeg
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## The showcase

Five short clips, one per behavior per lighting condition:

| clip | lighting | what it shows |
|------|----------|---------------|
| Locomotion_day | color | civet circling, away from the tray |
| Eating_day | color | civet head-down at the tray (tray rule) |
| Resting_day | color | civet lying still (motion gate keeps it Inactive) |
| Eating_night | infrared | civet eating at the tray — CNN reads the posture, no tray box (camera moved at night) |
| Locomotion_night | infrared | civet walking at night (lower detect threshold; the fast IR walker gives some gaps) |

Each prints its observation sheet + an annotated preview, and saves both to Drive at
`outputs/behavior_showcase/`.

In [ ]:
# 2) Run the showcase
import subprocess, shutil, datetime as dt, imageio_ffmpeg
from base64 import b64encode
from IPython.display import HTML, display
import pandas as pd
from pathlib import Path

FF = imageio_ffmpeg.get_ffmpeg_exe()
CIVET_MODEL    = drive_root / "models" / "civet_bowl_seg_best.pt"   # YOLO-seg civet(+bowl) detector
BEHAVIOR_MODEL = drive_root / "models" / "model.pt"                 # 3-class behavior ResNet
TRAY_BOX       = "706,492,808,573"                                 # Camera-3 DAYTIME food-tray box
DAY_VIDEO      = drive_root / "data" / "raw" / "cam3" / "A03_20260405140327.mp4"
NIGHT_VIDEO    = drive_root / "data" / "raw" / "cam3" / "A03_20260405000000.mp4"
for p in (CIVET_MODEL, BEHAVIOR_MODEL, DAY_VIDEO, NIGHT_VIDEO):
    assert p.exists(), f"Missing on Drive: {p}"

SHOWCASE_DIR = drive_root / "outputs" / "behavior_showcase"          # results persist here
SHOWCASE_DIR.mkdir(parents=True, exist_ok=True)

DAY = "2026-04-05 14:03:27"; NIGHT = "2026-04-05 00:00:00"         # recording start clocks
# (name, video, start_s, end_s, tray_box or None, recording_start, classify_conf)
SHOWCASE = [
    ("Locomotion_day",   DAY_VIDEO,   7120,  7148,  TRAY_BOX, DAY,   0.50),
    ("Eating_day",       DAY_VIDEO,   6345,  6390,  TRAY_BOX, DAY,   0.50),
    ("Resting_day",      DAY_VIDEO,   7640,  7700,  TRAY_BOX, DAY,   0.50),
    ("Eating_night",     NIGHT_VIDEO, 2145,  2190,  None,     NIGHT, 0.50),
    ("Locomotion_night", NIGHT_VIDEO, 11600, 11680, None,     NIGHT, 0.35),  # fast IR walker: lower conf
]

def _show(path, title):
    data = b64encode(Path(path).read_bytes()).decode()
    display(HTML(f"<h3>{title}</h3>"
                 f'<video width="640" controls src="data:video/mp4;base64,{data}"></video>'))

for name, video, a, b, tray, rec_start, cconf in SHOWCASE:
    clip = f"/content/show_{name}.mp4"; outdir = f"/content/show_out_{name}"
    subprocess.run([FF, "-y", "-v", "error", "-ss", str(a), "-to", str(b),
                    "-i", str(video), "-c:v", "libx264", "-pix_fmt", "yuv420p", clip], check=True)
    clk = (dt.datetime.fromisoformat(rec_start) + dt.timedelta(seconds=a)).strftime("%Y-%m-%d %H:%M:%S")
    cmd = ["python", str(PIPE / "run_tracked_behavior.py"), "--video", clip,
           "--civet-model", str(CIVET_MODEL), "--behavior-model", str(BEHAVIOR_MODEL),
           "--out", outdir, "--cctv-start", clk, "--classify-conf", str(cconf)]
    if tray:
        cmd += ["--tray-box", tray]
    subprocess.run(cmd, check=True)
    h264 = f"{outdir}/preview_h264.mp4"
    subprocess.run([FF, "-y", "-v", "error", "-i", f"{outdir}/tracked_preview.mp4",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", h264], check=True)
    shutil.copy2(h264, SHOWCASE_DIR / f"{name}_preview.mp4")
    shutil.copy2(f"{outdir}/observation_sheet.csv", SHOWCASE_DIR / f"{name}_observation_sheet.csv")
    _show(h264, name)
    print(f"{name} — observation sheet (time, status, behavior):")
    display(pd.read_csv(f"{outdir}/observation_sheet.csv"))

print("Saved previews + observation sheets to Drive:", SHOWCASE_DIR)